### Colab Instructions

- **Files and Libraries Are Handled Automatically**  
  The dataset, model checkpoint, `compute_cost.py`, and all required libraries will be automatically downloaded and installed when you run the notebook.

- **Enable GPU Acceleration**  
  Go to **Runtime** → **Change runtime type**, and select **GPU** (e.g., **T4 GPU**) as the hardware accelerator.

- **Run the Notebook**  
  Click **Runtime** → **Run all** to execute all cells sequentially.

- **Logging with Weights and Biases**  
  The notebook will prompt you to paste your API token. You can obtain the token by creating a free account at [Weights and Biases](https://wandb.ai/site/).

- **Working Directory**  
  The working directory is set to `/content`.

- **Runtime Duration**  
  Running the full notebook will take approximately 45 minutes.

# Advanced Sound Event Detection Tutorial

In this tutorial, you will learn how to:
- Create a train/validation/test split  
- Evaluate classifiers using standard metrics (e.g., precision, recall, f1-score)  
- Compute segment-level costs based on classifier output  
- Establish a simple baseline
- Train and assess a logistic regression model on audio embeddings  
- Build and evaluate a bidirectional RNN for sequence modeling  
- Compare cost performance across baseline, logistic regression, and RNN models on the test set
- Run inference on the customer's secret test set and store the predictions

In [1]:
 # Install required packages
 # !pip install --quiet numpy pandas matplotlib scikit-learn torch torchvision torchaudio pytorch-lightning wandb rich ipywidgets tabulate tqdm

In [1]:
# TODO: Keep compute_cost.py and mlpc2025_dataset in the working directory where this notebook is located

import os
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
    RichProgressBar
)
from pytorch_lightning.loggers import WandbLogger
from tqdm import tqdm
from huggingface_hub import snapshot_download, hf_hub_download
import zipfile
import shutil

# imports for custom methods
import scipy.signal
from collections import defaultdict

In [2]:
# download the compute_cost.py file
pyfile_path = hf_hub_download(
   repo_id="fschmid56/mlpc2025_dataset",
   filename="compute_cost.py",
   repo_type="dataset"
)

# put cost matrix here for easy of access
COST_MATRIX = {
    "Speech":         {"TP": 0,  "FP": 1,  "TN": 0, "FN": 5},
    "Dog Bark":       {"TP": 0,  "FP": 1,  "TN": 0, "FN": 5},
    "Rooster Crow":   {"TP": 0,  "FP": 1,  "TN": 0, "FN": 5},
    "Shout":          {"TP": 0,  "FP": 2,  "TN": 0, "FN": 10},
    "Lawn Mower":     {"TP": 0,  "FP": 3,  "TN": 0, "FN": 15},
    "Chainsaw":       {"TP": 0,  "FP": 3,  "TN": 0, "FN": 15},
    "Jackhammer":     {"TP": 0,  "FP": 3,  "TN": 0, "FN": 15},
    "Power Drill":    {"TP": 0,  "FP": 3,  "TN": 0, "FN": 15},
    "Horn Honk":      {"TP": 0,  "FP": 3,  "TN": 0, "FN": 15},
    "Siren":          {"TP": 0,  "FP": 3,  "TN": 0, "FN": 15},
}

# move to current working directory (/content)
shutil.copy(pyfile_path, os.getcwd() + "/compute_cost.py")

# import required functions
from compute_cost import CLASSES as TARGET_CLASSES # Speech, Shout, Chainsaw, Jackhammer, Lawn Mower, Power Drill, Dog Bark, Rooster Crow, Horn Honk, Siren
from compute_cost import (
    aggregate_targets,
    get_ground_truth_df,
    get_segment_prediction_df,
    check_dataframe,
    total_cost
)

## Download and prepare MLPC2025 Dataset

In [3]:
# # Step 1: Download the ZIP file from HF Hub
# zip_path = hf_hub_download(
#    repo_id="fschmid56/mlpc2025_dataset",   # your dataset repo
#    filename="mlpc2025_dataset.zip",        # your uploaded ZIP file
#    repo_type="dataset"                     # specify that it's a dataset repo
# )
#
# print(f"✅ ZIP downloaded: {zip_path}")

In [4]:
# ## Step 2: Extract the ZIP
extract_path = os.getcwd() + "/mlpc2025_dataset"
# os.makedirs(extract_path, exist_ok=True)
#
# # Check if already extracted
# if not os.path.exists(os.path.join(extract_path, "data")):  # assuming 'data/' is inside the zip
#    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#        zip_ref.extractall(extract_path)
#    print(f"✅ Dataset extracted to {extract_path}")
# else:
#    print(f"✅ Dataset already extracted at {extract_path}")
print(extract_path)

C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Projects\MLPC\MLPC Project Task 3\Challenge Tutorial/mlpc2025_dataset


In [5]:
# Step 3: Set your DATASET_PATH
DATASET_PATH = os.path.join(extract_path, "data")  # because you zipped the 'data' folder
print(f"✅ DATASET_PATH set to {DATASET_PATH}")

# Quick check
print("Files in DATASET_PATH:", os.listdir(DATASET_PATH))

✅ DATASET_PATH set to C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Projects\MLPC\MLPC Project Task 3\Challenge Tutorial/mlpc2025_dataset\data
Files in DATASET_PATH: ['.cache', 'annotations.csv', 'audio', 'audio_features', 'customer_test_data', 'labels', 'metadata.csv']


In [112]:
METADATA_CSV = os.path.join(DATASET_PATH, 'metadata.csv')
ANNOTATIONS_CSV = os.path.join(DATASET_PATH, 'annotations.csv')
AUDIO_DIR = os.path.join(DATASET_PATH, 'audio')
AUDIO_FEATURES_DIR = os.path.join(DATASET_PATH, 'audio_features')
LABELS_DIR = os.path.join(DATASET_PATH, 'labels')

METADATA = pd.read_csv(METADATA_CSV)
DEV_SET_FILES = METADATA['filename']

CUSTOMER_DATASET_PATH = os.path.join(DATASET_PATH, 'customer_test_data')
CUSTOMER_AUDIO_DIR = os.path.join(CUSTOMER_DATASET_PATH, 'audio')
CUSTOMER_AUDIO_FEATURES_DIR = os.path.join(CUSTOMER_DATASET_PATH, 'audio_features')
CUSTOMER_METADATA_CSV = os.path.join(CUSTOMER_DATASET_PATH, 'metadata.csv')
CUSTOMER_METADATA = pd.read_csv(CUSTOMER_METADATA_CSV)

DATA_SUBSAMPLE = 4938  # works with available RAM in Colab # TODO: VERY SMALL TEST for testing, MAX: 4938

## Create the Data Split

In [7]:
def read_files(file_names, classes, features_dir=AUDIO_FEATURES_DIR, labels_dir=LABELS_DIR):
    """
    Loads features and binary labels for a list of files.

    Returns:
        X: list of np.ndarrays, each of shape (num_frames, num_features)
        Y: dict of lists of np.ndarrays, each of shape (num_frames,)
    """
    X = []
    Y = {c: [] for c in classes} if labels_dir is not None else None

    for fname in file_names:
        base = os.path.splitext(fname)[0]

        # Load features
        feat_path = os.path.join(features_dir, base + '.npz')
        features = np.load(feat_path)['embeddings']  # shape: (T, D)
        X.append(features)

        if labels_dir is not None:
            # Load labels
            label_path = os.path.join(labels_dir, base + '_labels.npz')
            labels = np.load(label_path)

            for c in classes:
                label_array = labels[c]  # shape: (T, num_annotators)
                binary_labels = (np.max(label_array, axis=1) > 0).astype(int)
                Y[c].append(binary_labels)  # shape: (T,)

    return X, Y

In [8]:
# Get filenames for split based on filenames
all_files = DEV_SET_FILES.unique()

# First split: 60% train, 40% temp (val + test)
train_files, temp_files = train_test_split(
    all_files, test_size=0.4, random_state=42, shuffle=True
)

# Second split: 50% val, 50% test from the remaining 40%
val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=42, shuffle=True
)

train_files = train_files[:DATA_SUBSAMPLE]
# train_files = train_files

print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

# Load features and labels
X_train, Y_train = read_files(train_files, TARGET_CLASSES)
X_val, Y_val = read_files(val_files, TARGET_CLASSES)
X_test, Y_test = read_files(test_files, TARGET_CLASSES)

#
# X_val = X_val[:DATA_SUBSAMPLE]
# Y_val = {c: Y_val[c][:DATA_SUBSAMPLE] for c in TARGET_CLASSES}
#
# X_test = X_test[:DATA_SUBSAMPLE]
# Y_test = {c: Y_test[c][:DATA_SUBSAMPLE] for c in TARGET_CLASSES}
#
# print(f"After subsample: Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 4938, Val: 1646, Test: 1646


## Evaluation Functions (Metrics & Cost)

In [9]:
# Flatten: Each frame is a sample
def flatten_for_framewise_classification(X, Y_class):
    X_flat = np.concatenate(X)  # shape: (total_frames, num_features)
    Y_flat = np.concatenate(Y_class)  # shape: (total_frames,)
    return X_flat, Y_flat

In [10]:
def evaluate_classifiers(
    classes: list[str],
    Y_val: dict[str, list[np.ndarray]],
    X_val: list[np.ndarray] = None,
    inference_funcs: dict[str, callable] = None,
    Y_pred: dict[str, list[np.ndarray]] = None
) -> tuple[dict[str, list[np.ndarray]], dict[str, dict]]:
    """
    Evaluates per-frame binary classifiers and computes metrics per class.
    Uses either computed predictions or given inference functions.

    Args:
        classes: List of class names to evaluate.
        Y_val: Dict mapping class names to lists of ground-truth (T,) binary arrays.
        X_val: List of input feature arrays, one per validation file. Required if Y_pred not given.
        inference_funcs: Dict mapping class names to binary inference functions.
        Y_pred: Dict with precomputed predictions (same format as Y_val).

    Returns:
        metrics: Dict[class → {'balanced_accuracy', 'precision', 'recall', 'f1'}].
    """

    if Y_pred is None:
        assert inference_funcs is not None and X_val is not None, "If 'Y_pred' is not given, 'inference_funcs' \
                                                                    and 'X_val' must be given."

    Y_val_preds = {}
    metrics     = {}

    for cls in classes:
        # use predictions if given, else infer
        if Y_pred and cls in Y_pred:
            preds_per_file = Y_pred[cls]
        else:
            infer = inference_funcs[cls]
            preds_per_file = [infer(x_file) for x_file in X_val]
        Y_val_preds[cls] = preds_per_file

        # flatten to compute metrics
        y_true = np.concatenate(Y_val[cls])
        y_pred = np.concatenate(preds_per_file)

        metrics[cls] = {
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision":         precision_score(y_true, y_pred, zero_division=0),
            "recall":            recall_score(y_true, y_pred, zero_division=0),
            "f1":                f1_score(y_true, y_pred, zero_division=0),
        }

    return metrics

In [11]:
def evaluate_cost(
    val_files: list[str],
    dataset_path: str,
    classes: list[str],
    X_val: list[np.ndarray] = None,
    inference_funcs: dict[str, callable] = None,
    Y_pred: dict[str, list[np.ndarray]] = None
):
    """
    Computes segment-level cost based on predictions and ground truth.
    Uses either computed predictions or given inference functions.

    Args:
        val_files: List of filenames corresponding to X_val.
        dataset_path: Path to dataset root (used for loading ground truth).
        classes: List of class names to evaluate.
        X_val: List of input feature arrays, one per validation file. Required if Y_pred not given.
        inference_funcs: Dict mapping class names to binary inference functions.
        Y_pred: Dict with precomputed predictions (class → list of (T,) arrays).

    Returns:
        total: Total cost across all validation files.
        breakdown: Dict[class → segment-level cost].
    """

    if Y_pred is None:
        assert inference_funcs is not None and X_val is not None, "If 'Y_pred' is not given, 'inference_funcs' \
                                                                    and 'X_val' must be given."

    # 0) frame-wise predictions (per class)
    if Y_pred is None:
        Y_pred = {
            cls: [infer(x_file) for x_file in X_val]
            for cls, infer in inference_funcs.items()
        }

    # 1) restructure to filename -> class -> (T,) array
    preds_by_file = {}
    for i, fname in enumerate(val_files):
        preds_by_file[fname] = {
            cls: Y_pred[cls][i] for cls in classes
        }

    # 2) segment-level aggregation using compute_cost
    pred_df = get_segment_prediction_df(
        predictions=preds_by_file,
        class_names=classes
    )

    # 3) load & aggregate ground truth using compute_cost
    gt_df = get_ground_truth_df(val_files, dataset_path)

    # 4) sanity checks from compute_cost
    check_dataframe(pred_df, dataset_path)
    check_dataframe(gt_df, dataset_path)

    # 5) compute cost
    total, breakdown = total_cost(pred_df, gt_df)

    return total, breakdown

In [ ]:
def calculate_cost_for_class(y_true, y_pred, class_name):
    """ Calculates the cost for a single class based on frame-level predictions. """
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    cost = (
        COST_MATRIX[class_name]["TP"] * tp +
        COST_MATRIX[class_name]["FP"] * fp +
        COST_MATRIX[class_name]["TN"] * tn +
        COST_MATRIX[class_name]["FN"] * fn
    )

    return cost

def tune_thresholds(Y_val, Y_pred_probs, classes):
    """ Finds optimal threshold for each class to minimize cost.
    Returns a dict mapping class name to its optimal threshold.
    """

    optimal_thresholds = {}

    print("Tuning thresholds for each class...")
    for cls in classes:
        # Flatten list of arrays into single long arrays for efficient computation
        y_true_flat = np.concatenate(Y_val[cls])
        y_probs_flat = np.concatenate(Y_pred_probs[cls])

        best_threshold = 0.5
        lowest_cost = float('inf')

        # iterate over a range of thresholds to find the best one
        for threshold in np.arange(0.05, 0.95, 0.01):
            y_pred_binary = (y_probs_flat > threshold).astype(int)
            current_cost = calculate_cost_for_class(y_true_flat, y_pred_binary, cls)

            if current_cost < lowest_cost:
                lowest_cost = current_cost
                best_threshold = threshold

        optimal_thresholds[cls] = best_threshold
        print(f" - Optimal threshold for {cls}: {best_threshold:.3f} (cost: {lowest_cost:.3f})")


    return optimal_thresholds

# MORE ADVANCED FUNCTIONS
Used at the end of the notebook

In [101]:
# POST-PROCESSING FUNCTIONS
def apply_median_filter(predictions_by_file, classes, kernel_size=5):
    """
    Applies a median filter to smooth out predictions.
    This should be applied to PROBABILITIES before thresholding.
    """
    print(f"Applying median filter with kernel size {kernel_size}...")

    # This line is crucial. It must be defaultdict(dict).
    filtered_preds = defaultdict(dict)

    for fname, class_preds in predictions_by_file.items():
        for cls in classes:
            # Ensure the class exists in the predictions for the file
            if cls in class_preds:
                probs = class_preds[cls]
                # Ensure kernel size is not larger than the array length
                k = min(kernel_size, len(probs))
                if k % 2 == 0: k -= 1 # kernel size must be odd

                if k > 0:
                  filtered_preds[fname][cls] = scipy.signal.medfilt(probs, kernel_size=k)
                else:
                  # If kernel size is 0 or less, just copy the original probabilities
                  filtered_preds[fname][cls] = probs

    return filtered_preds

def apply_duration_filtering(predictions_by_file, classes, min_duration_frames=3):
    """ Removes short, spurious positive predictions.
    APPLY THIS TO BINARY PREDICTIONS AFTER THRESHOLDING
    Returns predictions with short events removed as a dict """
    print(f"Applying duration filtering with min duration {min_duration_frames} frames...")
    filtered_preds = defaultdict(dict)
    for fname, class_preds in predictions_by_file.items():
        for cls in classes:
            binary_preds = class_preds[cls].copy()

            # Find start and end of positive segments
            diff = np.diff(np.concatenate(([0], binary_preds, [0])))
            starts = np.where(diff == 1)[0]
            ends = np.where(diff == -1)[0]

            for start, end in zip(starts, ends):
                duration = end - start
                if duration < min_duration_frames:
                    binary_preds[start:end] = 0 # set short segments to 0

            filtered_preds[fname][cls] = binary_preds
    return filtered_preds


In [14]:
# ENSEMBLE FUNCTIONS
def ensemble_predictions(list_of_prediction_dicts, classes, weights=None):
    """ Averages predictions from multiple models.
     Operates on probability predictions. Returns: A single dictionary with ensembled predictions.
     """

    if not list_of_prediction_dicts:
        return {}

    if weights is None:
        weights = [1.0 / len(list_of_prediction_dicts)] * len(list_of_prediction_dicts) # uniform weights

    print(f"Ensembling {len(list_of_prediction_dicts)} models with weights {weights}...")

    ensembled_preds = defaultdict(dict)
    filenames = list_of_prediction_dicts[0].keys()

    for fname in filenames:
        for cls in classes:
            # Get predictions for the current file and class from all models
            model_probs = [preds[fname][cls] for preds in list_of_prediction_dicts]

            # weighted average
            avg_prob = np.average(model_probs, axis=0, weights=weights)

            ensembled_preds[fname][cls] = avg_prob

    return ensembled_preds

## Most-Frequent Label Baseline

In [16]:
def baseline_most_frequent(
    Y_train: dict[str, list[np.ndarray]],
    classes: list[str]
) -> dict[str, callable]:
    """
    Returns inference functions that always predict each class’s majority label.
    """
    inference_funcs = {}
    for cls in classes:
        all_frames = np.concatenate(Y_train[cls])
        most_freq_label  = int(np.mean(all_frames) >= 0.5)
        # inference func ignores features, just returns most frequent label per frame
        inference_funcs[cls] = lambda x, ml=most_freq_label: np.full(x.shape[0], ml, dtype=int)
    return inference_funcs

# 1) Create baseline’s inference functions
bl_inference_funcs = baseline_most_frequent(Y_train, TARGET_CLASSES)

In [17]:
# metrics for most-frequent label baseline
val_metrics = evaluate_classifiers(
    classes=TARGET_CLASSES,
    X_val=X_val,
    Y_val=Y_val,
    inference_funcs=bl_inference_funcs
)

df = pd.DataFrame(val_metrics).T.round(3)
df.columns = ["BAcc", "Precision", "Recall", "F1"]
print(tabulate(df, headers='keys', tablefmt='github'))

|              |   BAcc |   Precision |   Recall |   F1 |
|--------------|--------|-------------|----------|------|
| Speech       |    0.5 |           0 |        0 |    0 |
| Shout        |    0.5 |           0 |        0 |    0 |
| Chainsaw     |    0.5 |           0 |        0 |    0 |
| Jackhammer   |    0.5 |           0 |        0 |    0 |
| Lawn Mower   |    0.5 |           0 |        0 |    0 |
| Power Drill  |    0.5 |           0 |        0 |    0 |
| Dog Bark     |    0.5 |           0 |        0 |    0 |
| Rooster Crow |    0.5 |           0 |        0 |    0 |
| Horn Honk    |    0.5 |           0 |        0 |    0 |
| Siren        |    0.5 |           0 |        0 |    0 |


In [18]:
# cost for most-frequent label baseline
total, breakdown = evaluate_cost(
    val_files=val_files,
    dataset_path=DATASET_PATH,
    classes=TARGET_CLASSES,
    X_val=X_val,
    inference_funcs=bl_inference_funcs
)

df = pd.DataFrame({cls: {"Avg. Cost per minute": round(m["cost"], 4)} for cls, m in breakdown.items()}).T
print(f"Total average cost per minute: {total:.4f}\n")
print(tabulate(df, headers="keys", tablefmt="github"))

Total average cost per minute: 108.8553

|              |   Avg. Cost per minute |
|--------------|------------------------|
| Speech       |                24.8092 |
| Shout        |                 9.4118 |
| Chainsaw     |                 6.6057 |
| Jackhammer   |                 7.4642 |
| Lawn Mower   |                 7.7266 |
| Power Drill  |                14.2369 |
| Dog Bark     |                 7.0986 |
| Rooster Crow |                 0.3816 |
| Horn Honk    |                12.7107 |
| Siren        |                18.4102 |


### Logistic Regression

In [19]:
def train_logistic_regression(
    X_train: list[np.ndarray],
    Y_train: dict[str, list[np.ndarray]],
    classes: list[str]
) -> dict[str, callable]:
    """
    Trains one scaler+logistic-regression per class and returns a dict of
    inference functions. Each function takes a (T, D) feature array and
    returns a (T,) array of {0,1} predictions.
    """
    inference_funcs = {}
    for cls in classes:
        # prepare frame-wise training data
        X_tr, y_tr = flatten_for_framewise_classification(X_train, Y_train[cls])

        # fit scaler and model
        scaler = StandardScaler().fit(X_tr)
        X_tr_scaled = scaler.transform(X_tr)
        clf = LogisticRegression(
            max_iter=100,
            class_weight='balanced',
            random_state=42
        ).fit(X_tr_scaled, y_tr)

        # define and store the joined inference function
        def make_inference(scaler, clf):
            return lambda x: clf.predict(scaler.transform(x))

        inference_funcs[cls] = make_inference(scaler, clf)

    return inference_funcs

lr_inference_funcs = train_logistic_regression(
   X_train, Y_train, TARGET_CLASSES
)

C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [20]:
val_metrics = evaluate_classifiers(
   classes=TARGET_CLASSES,
   X_val=X_val,
   Y_val=Y_val,
   inference_funcs=lr_inference_funcs
)

df = pd.DataFrame(val_metrics).T.round(3)
df.columns = ["BAcc", "Precision", "Recall", "F1"]
print(tabulate(df, headers='keys', tablefmt='github'))

|              |   BAcc |   Precision |   Recall |    F1 |
|--------------|--------|-------------|----------|-------|
| Speech       |  0.919 |       0.633 |    0.89  | 0.74  |
| Shout        |  0.761 |       0.227 |    0.554 | 0.322 |
| Chainsaw     |  0.899 |       0.535 |    0.805 | 0.643 |
| Jackhammer   |  0.681 |       0.331 |    0.37  | 0.349 |
| Lawn Mower   |  0.752 |       0.387 |    0.513 | 0.441 |
| Power Drill  |  0.735 |       0.224 |    0.502 | 0.31  |
| Dog Bark     |  0.91  |       0.63  |    0.831 | 0.717 |
| Rooster Crow |  0.795 |       0.482 |    0.591 | 0.531 |
| Horn Honk    |  0.784 |       0.272 |    0.591 | 0.373 |
| Siren        |  0.854 |       0.766 |    0.714 | 0.739 |


In [21]:
# inference_funcs from train_logistic_regression_inference(...)
total, breakdown = evaluate_cost(
   val_files=val_files,
   dataset_path=DATASET_PATH,
   classes=TARGET_CLASSES,
   X_val=X_val,
   inference_funcs=lr_inference_funcs
)

df = pd.DataFrame({cls: {"Avg. Cost per minute": round(m["cost"], 4)} for cls, m in breakdown.items()}).T
print(f"Total average cost per minute: {total:.4f}\n")
print(tabulate(df, headers="keys", tablefmt="github"))

Total average cost per minute: 56.2973

|              |   Avg. Cost per minute |
|--------------|------------------------|
| Speech       |                 5.3561 |
| Shout        |                 7.9905 |
| Chainsaw     |                 2.2655 |
| Jackhammer   |                 5.7758 |
| Lawn Mower   |                 5.5469 |
| Power Drill  |                12.3816 |
| Dog Bark     |                 1.6948 |
| Rooster Crow |                 0.2051 |
| Horn Honk    |                 9.601  |
| Siren        |                 5.4801 |


# Bidirectional Gated Recurrent Unit

Can a recurrent neural network, which models temporal dependencies across frames, outperform logistic regression, which treats each frame independently, in sound event detection?

We will implement the required ingredients in the following order:
* Dataset
* DataModule
* RNN Model
* PyTorch Lightning Module
* Hyperparameter Configuration
* Logging via Weights & Biases
* Callbacks
* PyTorch Lightning Trainer

## Dataset



In [22]:
class SequenceDataset(Dataset):
    """
    Dataset for sequence modeling tasks with optional per-frame binary labels.

    Args:
        X: List of input feature arrays (T_i, D), one per file.
        Y: Optional dict[class → list of (T_i,) label arrays], one per file and class.
        classes: List of class names to extract from Y.
        filenames: List of filenames corresponding to each input.

    Returns:
        Each item is a tuple:
        - (features, labels, filename): if Y is provided
        - (features, filename): if Y is None
    """
    def __init__(self, X, Y, classes, filenames):
      # in colab with limited RAM, we convert our files to
      # tensors only in __getitem__
      self.X = X  # Keep X as a list of np.ndarrays
      self.Y = Y
      self.classes = classes
      self.filenames = filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.X[idx], dtype=torch.float32)  # Convert on access
        if self.Y is not None:
            y_tensor = torch.stack([
                torch.tensor(self.Y[c][idx], dtype=torch.long) for c in self.classes
            ], dim=1)
            return x_tensor, y_tensor, self.filenames[idx]
        else:
            return x_tensor, self.filenames[idx]

In [23]:
ds = SequenceDataset(X_train, Y_train, TARGET_CLASSES, train_files)
feat0, label0, file0 = ds[0]
print("SequenceDataset[0] -> feature shape:", feat0.shape,
      "\nlabel shape:", label0.shape,
      "\nfile[0]:", file0)

SequenceDataset[0] -> feature shape: torch.Size([141, 768]) 
label shape: torch.Size([141, 10]) 
file[0]: 770482.mp3


In [24]:
# collate_fn used to create batches from the individual dataset items
def collate_fn(batch):
    if len(batch[0]) == 3:
        Xs, Ys, filenames = zip(*batch)
        lengths = torch.tensor([x.size(0) for x in Xs], dtype=torch.long)
        X_padded = pad_sequence(Xs, batch_first=True)
        Y_padded = pad_sequence(Ys, batch_first=True)
        return X_padded, Y_padded, lengths, list(filenames)
    elif len(batch[0]) == 2:
        Xs, filenames = zip(*batch)
        lengths = torch.tensor([x.size(0) for x in Xs], dtype=torch.long)
        X_padded = pad_sequence(Xs, batch_first=True)
        return X_padded, lengths, list(filenames)
    else:
        raise ValueError("Unexpected batch format: expected 2 or 3 elements per item.")

In [25]:
batch = [ds[i] for i in range(32)]
X_pad, Y_pad, lengths, filenames = collate_fn(batch)

print("collate_fn -> X_padded:", X_pad.shape,
      "\nY_padded:", Y_pad.shape,
      "\nlengths:", lengths,
      "\nfilenames:", filenames[:3], "...")

collate_fn -> X_padded: torch.Size([32, 249, 768]) 
Y_padded: torch.Size([32, 249, 10]) 
lengths: tensor([141, 242, 249, 208, 139, 140, 228, 161, 206, 184, 156, 217, 149, 162,
        152, 128, 183, 127, 137, 234, 187, 166, 212, 239, 214, 128, 143, 159,
        142, 210, 210, 195]) 
filenames: ['770482.mp3', '649348.mp3', '558402.mp3'] ...


## DataModule

A `LightningDataModule` which organizes **all data loading logic** in one place.

Implements the following core API.

| Method                 | Purpose                                |
|------------------------|----------------------------------------|
| `__init__()`           | Save paths, batch size, classes, etc.  |
| `setup(stage)`         | Prepare datasets (train/val/test)      |
| `train_dataloader()`   | Return DataLoader for training         |
| `val_dataloader()`     | Return DataLoader for validation       |
| `test_dataloader()`    | Return DataLoader for testing          |

In [26]:
# Calculate Positive Weights for the Loss Function
# Weight for positive samples is the ratio of FN cost to FP cost
pos_weights = torch.tensor([
    COST_MATRIX[c]["FN"] / COST_MATRIX[c]["FP"]
    for c in TARGET_CLASSES
])

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DataModule is used by pytorch lightning
class SEDDataModule(pl.LightningDataModule):
    def __init__(self,
                 X_train, Y_train, train_files,
                 X_val,   Y_val,   val_files,
                 X_test,  Y_test,  test_files,
                 classes,
                 batch_size=32,
                 num_workers=2):
        super().__init__()
        self.X_train, self.Y_train, self.train_files = X_train, Y_train, train_files
        self.X_val,   self.Y_val,   self.val_files   = X_val,   Y_val,   val_files
        self.X_test,  self.Y_test,  self.test_files  = X_test,  Y_test,  test_files
        self.classes     = classes
        self.batch_size  = batch_size
        self.num_workers = num_workers



    def setup(self, stage=None):
        self.train_ds = SequenceDataset(self.X_train, self.Y_train, self.classes, self.train_files)
        self.val_ds   = SequenceDataset(self.X_val,   self.Y_val,   self.classes, self.val_files)
        self.test_ds  = SequenceDataset(self.X_test,  self.Y_test,  self.classes, self.test_files)

    def train_dataloader(self):
        return DataLoader(self.train_ds,
                          batch_size=self.batch_size,
                          shuffle=True,
                          collate_fn=collate_fn,
                          num_workers=self.num_workers,
                          persistent_workers=False) # TODO: IMPORTANT, true didn't work

    def val_dataloader(self):
        return DataLoader(self.val_ds,
                          batch_size=self.batch_size,
                          shuffle=False,
                          collate_fn=collate_fn,
                          num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_ds,
                          batch_size=self.batch_size,
                          shuffle=False,
                          collate_fn=collate_fn,
                          num_workers=self.num_workers)

In [28]:
from tqdm.notebook import tqdm

dm = SEDDataModule(
    X_train=X_train, Y_train=Y_train, train_files=train_files,
    X_val=X_val,     Y_val=Y_val,     val_files=val_files,
    X_test=X_test,   Y_test=Y_test,   test_files=test_files,
    classes=TARGET_CLASSES,
    batch_size=32,
    num_workers=0
)

dm.setup()
loader = dm.train_dataloader()
for batch in tqdm(loader):
    pass
X_batch, Y_batch, len_batch, filenames = next(iter(loader))
print("DataModule batch -> X:", X_batch.shape,
      "\nY:", Y_batch.shape,
      "\nlengths:", len_batch,
      "\nfilenames:", filenames[:3], "...")

  0%|          | 0/155 [00:00<?, ?it/s]

DataModule batch -> X: torch.Size([32, 249, 768]) 
Y: torch.Size([32, 249, 10]) 
lengths: tensor([130, 130, 173, 126, 166, 144, 191, 211, 186, 214, 221, 204, 135, 225,
        181, 149, 204, 165, 228, 249, 183, 174, 147, 139, 165, 157, 166, 176,
        148, 186, 167, 221]) 
filenames: ['636907.mp3', '127708.mp3', '667744.mp3'] ...


### Bidirectional RNN

In [29]:
class BiGRUClassifier(nn.Module):
    """
    Bidirectional GRU classifier with a linear output layer.

    Args:
        input_dim: Input feature dimension (D).
        hidden_dim: Hidden size per GRU direction.
        num_layers: Number of stacked GRU layers.
        num_classes: Number of output classes (C).

    Input:
        x: Tensor of shape (B, T, D) — batch of padded sequences.
        lengths: Tensor of shape (B,) — actual lengths before padding.

    Returns:
        logits: Tensor of shape (B, T, C) — class scores for each time step.
    """
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # x: (B, T, D), lengths: (B,)
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        # out: (B, T, 2*hidden_dim)
        logits = self.classifier(out)  # (B, T, num_classes)
        return logits

In [30]:
# Instantiate model
model = BiGRUClassifier(
    input_dim=X_batch.shape[-1],
    hidden_dim=1024,
    num_layers=2,
    num_classes=Y_batch.shape[-1]
)

# Forward pass
logits = model(X_batch, len_batch)

# Print shapes
print("Input X_batch shape:", X_batch.shape)       # (B, T_max, F)
print("Output logits shape:", logits.shape)         # (B, T_max, C)

Input X_batch shape: torch.Size([32, 249, 768])
Output logits shape: torch.Size([32, 249, 10])


### PyTorch Lightning Module: `SEDLightningModule`

The `LightningModule` wraps your model and training logic, abstracting away boilerplate code and handling key training steps automatically.  
It implements a **standardized API** to define how your model should behave during training, validation, testing, and prediction.

- **`__init__`**: initializes the model (`BiGRUClassifier`), loss function, and validation and test buffer storage, and sets important attributes (e.g., lr, threshold).
- **`forward(x, lengths)`**: forward pass through the GRU model.
- **`predict_step(batch, batch_idx)`**: applies sigmoid + thresholding, slices off padding → returns predictions.
- **`training_step(batch, batch_idx)`**: handles training logic.
- **`validation_step(batch, batch_idx)`**: handles validation logic.
- **`on_validation_epoch_end()`**: aggregates validation results after each epoch.
- **`configure_optimizers()`**: defines the optimizer (Adam).

In [31]:
class SEDLightningModule(pl.LightningModule):
    # def __init__(self, input_dim, hidden_dim, num_layers, classes, lr=1e-4, threshold=0.5):
    #     super().__init__()
    #     # Core model
    #     self.model = BiGRUClassifier(
    #         input_dim=input_dim,
    #         hidden_dim=hidden_dim,
    #         num_layers=num_layers,
    #         num_classes=len(classes)
    #     )
    #
    #     self.classes = classes
    #
    #     # Loss (we'll apply masking later, thus reduction='none')
    #     self.criterion = nn.BCEWithLogitsLoss(reduction='none')
    #     self.lr = lr
    #     self.threshold = threshold
    #
    #     self._val_preds   = {c: [] for c in self.classes}
    #     self._val_targets = {c: [] for c in self.classes}
    #     self._val_filenames = []

    def __init__(self, input_dim, hidden_dim, num_layers, classes, pos_weights, lr=1e-4, threshold=0.5):
        super().__init__()
        self.model = BiGRUClassifier(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_classes=len(classes)
        )
        self.classes = classes

        # Then use pos_weight to make the loss function aware of asymmetric costs
        self.criterion = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weights.to(device))

        self.lr = lr
        self.threshold = threshold
        self._val_preds   = {c: [] for c in self.classes}
        self._val_targets = {c: [] for c in self.classes}
        self._val_filenames = []

    def forward(self, x, lengths):
        return self.model(x, lengths)

    def predict_step(self, batch, batch_idx):
        # unpack batch (with or without labels)
        if len(batch) == 4:
            X, _, lengths, filenames = batch
        else:
            X, lengths, filenames = batch

        # 1) raw logits → probs → binary preds
        logits = self.model(X, lengths)
        probs  = torch.sigmoid(logits)
        preds  = (probs > self.threshold).int() # (B, T_max, C)

        # 2) remove padding
        batch_preds = [preds[b, :lengths[b]].cpu()
                      for b in range(X.size(0))]

        return {"filenames": filenames, "preds": batch_preds}

    # we will implement the processing steps one after the other in the following
    def training_step(self, batch, batch_idx):
        return self.process_training_step(batch, batch_idx)

    def validation_step(self, batch, batch_idx):
        return self.process_validation_step(batch, batch_idx)

    def on_validation_epoch_end(self):
        return self.process_validation_epoch_end()

    def test_step(self, batch, batch_idx):
        return self.process_test_step(batch, batch_idx)

    def on_test_epoch_end(self):
        return self.process_test_epoch_end()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

### PyTorch Lightning Module: `SEDLightningModule`

In the following, we will implement the missing functions:

- `process_training_step`:  
  computes masked BCE loss for each frame and logs the training loss.

- `process_validation_step`:  
  collects per-frame predictions and targets for later metric computation.

- `process_validation_epoch_end`:  
  aggregates predictions and targets, computes metrics, and logs results.

- `process_test_step`:  
  same as validation but used for test-time evaluation.

- `process_test_epoch_end`:  
  evaluates and logs performance after test epoch.

We will bind this functions to our `SEDLightningModule`.

### `process_training_step`

computes masked BCE loss for each frame and logs the training loss

In [32]:
def process_training_step(self, batch, batch_idx):
    X, Y, lengths, _ = batch      # X: (B, T, D), Y: (B, T, C), lengths: (B,)
    logits = self(X, lengths)     # calls self.forward, results in logits of shape (B, T, C)

    # raw per-element loss
    loss_raw = self.criterion(logits, Y.float())  # (B, T, C)

    # build mask to zero out padded frames
    mask = torch.arange(logits.size(1), device=logits.device)[None, :] < lengths[:, None]
    mask = mask.unsqueeze(-1).float()     # (B, T, 1)

    # apply mask and average
    loss = (loss_raw * mask).sum() / mask.sum()

    self.log('train/loss', loss, prog_bar=True, on_step=True, on_epoch=True, batch_size=X.size(0))
    return loss

# Bind it to the LightningModule
SEDLightningModule.process_training_step = process_training_step

### `process_validation_step`

computes masked BCE loss, logs it, and stores frame-level predictions and targets for aggreation in `process_validation_epoch_end`

In [33]:
def process_validation_step(self, batch, batch_idx):
    X, Y, lengths, filenames = batch      # X: (B, T, D), Y: (B, T, C), lengths: (B,)
    logits = self(X, lengths)             # calls self.forward, results in logits of shape (B, T, C)

    # Determine logging prefix
    prefix = "test" if self.trainer.testing else "val"

    # compute masked BCE loss
    loss_raw = self.criterion(logits, Y.float())     # (B, T, C)
    mask = torch.arange(logits.size(1), device=logits.device)[None, :] < lengths[:, None]
    mask = mask.unsqueeze(-1).float()                # (B, T, 1)
    loss = (loss_raw * mask).sum() / mask.sum()

    self.log(f'{prefix}/loss', loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=X.size(0))

    # store frame-wise preds & targets for epoch_end
    # frame-wise logits are thresholded here
    preds = (torch.sigmoid(logits) > self.threshold).long()     # (B, T, C)
    self._val_filenames.extend(filenames)

    for i, c in enumerate(self.classes):
        for b in range(X.size(0)):
            T = lengths[b]
            self._val_preds[c].append(preds[b, :T, i])
            self._val_targets[c].append(Y[b, :T, i])

    return loss

# Bind it to the LightningModule
SEDLightningModule.process_validation_step = process_validation_step

### `process_validation_epoch_end`

computes dataset metrics and cost and logs them at the end of a validation epoch.

In [34]:
def process_validation_epoch_end(self):
    # Determine current mode
    prefix = "test" if self.trainer.testing else "val"

    # --- 1) Convert buffered tensors to NumPy arrays ---
    preds_numpy = {
        cls: [p.cpu().numpy() for p in self._val_preds[cls]]
        for cls in self.classes
    }
    targets_numpy = {
        cls: [t.cpu().numpy() for t in self._val_targets[cls]]
        for cls in self.classes
    }

    # --- 2) Frame‐level metrics ---
    frame_metrics = evaluate_classifiers(
        classes=self.classes,
        Y_val=targets_numpy,
        Y_pred=preds_numpy
    )

    for cls, m in frame_metrics.items():
        self.log(f'{prefix}/{cls}_bacc',     m['balanced_accuracy'])
        self.log(f'{prefix}/{cls}_precision',m['precision'])
        self.log(f'{prefix}/{cls}_recall',   m['recall'])
        self.log(f'{prefix}/{cls}_f1',       m['f1'])

    # --- 3) Segment‐level cost ---
    total_cost, cost_breakdown = evaluate_cost(
        val_files=self._val_filenames,
        dataset_path=DATASET_PATH,
        classes=self.classes,
        Y_pred=preds_numpy
    )
    self.log(f'{prefix}/total_cost', total_cost, prog_bar=True)
    for cls, cls_cost in cost_breakdown.items():
        self.log(f"{prefix}/cost/{cls}", cls_cost["cost"], prog_bar=False)

    # --- 4) Clear buffers ---
    self._val_preds     = {c: [] for c in self.classes}
    self._val_targets   = {c: [] for c in self.classes}
    self._val_filenames = []


SEDLightningModule.process_validation_epoch_end = process_validation_epoch_end

### `process_test_step` and `process_test_epoch_end` ...

fortunately require the same logic as validation, so we can reuse `process_validation_step` and `process_validation_epoch_end`

In [35]:
# After you’ve attached the validation logic, simply reuse it for testing:

# Reuse the same step‐logic
SEDLightningModule.process_test_step = SEDLightningModule.process_validation_step

# Reuse the same epoch‐end logic
SEDLightningModule.process_test_epoch_end = SEDLightningModule.process_validation_epoch_end

### Hyperparameters

Key hyperparameters with reasonable initial values — most likely not guaranteed optimal.

In [74]:
hparams = dict(
    # not tuned by us - used out of the box
    input_dim      = X_batch.shape[-1],
    hidden_dim     = 256, # TODO: Initially 1024
    num_layers     = 1,
    lr             = 1e-4,
    batch_size     = 64,
    max_epochs     = 50,
    threshold      = 0.5,
    patience       = 10,         # Early-stopping patience TODO: Initially 5
)

### Callbacks

Callbacks are modular hooks that enable custom actions during training (e.g., saving checkpoints, early stopping, or logging), triggered at specific stages.

In [75]:
checkpoint_cb = ModelCheckpoint(
    monitor    = "val/total_cost",   # minimize cost
    mode       = "min",
    save_top_k = 1,                  # save top model on validation data
    filename   = "best-{epoch:02d}"
)

early_stop_cb = EarlyStopping(
    monitor  = "val/total_cost",
    mode     = "min",
    patience = hparams["patience"],
    verbose  = True
)

lr_monitor_cb = LearningRateMonitor(logging_interval="epoch")

# RichProgressBar generates minimal output compared to 'tqdm'
progress_bar_cb = RichProgressBar()

callbacks = [checkpoint_cb, early_stop_cb, lr_monitor_cb, progress_bar_cb]

### Logger

- [Weights & Biases (wandb)](https://wandb.ai/site/) is a powerful and free experiment tracking tool  
- lets you log metrics, visualize training runs, compare models  
- share results via an interactive online dashboard  
- integrates seamlessly with PyTorch Lightning

In [76]:
# wandb_logger = WandbLogger(
#     project     = "mlpc2025-sed",
#     name        = f"BiGRU-{hparams['hidden_dim']}x{hparams['num_layers']}",
#     config      = hparams
# )

# TODO: Run in offline mode
wandb_logger = WandbLogger(
    project     = "mlpc2025-sed",
    name        = f"BiGRU-{hparams['hidden_dim']}x{hparams['num_layers']}",
    config      = hparams,
    offline     = True  # <-- Run in offline mode
)

### Trainer

The `Trainer` is the central PyTorch Lightning component that orchestrates training, validation, and testing.

It brings everything together:
- The `SEDDataModule` provides the data.
- The `SEDLightningModule` defines the model and training logic.
- The `Trainer` handles the training loop, evaluation, logging, and callbacks.

In [77]:
dm = SEDDataModule(
    X_train=X_train, Y_train=Y_train, train_files=train_files,
    X_val=X_val,     Y_val=Y_val,     val_files=val_files,
    X_test=X_test,   Y_test=Y_test,   test_files=test_files,
    classes=TARGET_CLASSES,
    batch_size=hparams["batch_size"],
    num_workers=0 # TODO: This could be problematic, change to 0
)

model = SEDLightningModule(
    input_dim  = hparams["input_dim"],
    hidden_dim = hparams["hidden_dim"],
    num_layers = hparams["num_layers"],
    classes    = TARGET_CLASSES,
    lr         = hparams["lr"],
    pos_weights = pos_weights
)


trainer = pl.Trainer(
    accelerator             = "gpu", # TODO: Change to gpu
    devices                 = 1,
    max_epochs              = hparams["max_epochs"],
    callbacks               = callbacks,
    logger                  = wandb_logger,
    log_every_n_steps       = 10,
    deterministic           = True,
    check_val_every_n_epoch = 1,
    num_sanity_val_steps    = 0
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [78]:
!wandb login
#!wandb relog

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


### Let's train!

The following command launches training and validation, alternating across epochs. All results will be logged to Weights & Biases.
You can explore a completed training run here: https://api.wandb.ai/links/cp_tobi/plk26iu9

Checkpoints will stored in `mlpc2025-sed/<wandb_id>/checkpoints`.

In [79]:
trainer.fit(model, datamodule=dm)   # train and validate

C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\loggers\wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory .\mlpc2025-sed\6k652nc3\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BiGRUClassifier   │  1.6 M │ train │
│ 1 │ criterion │ BCEWithLogitsLoss │      0 │ train │
└───┴───────────┴───────────────────┴────────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 4                                                                                           
Modules in eval mode: 0

C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Output()

Metric val/total_cost improved. New best score: 89.099
Metric val/total_cost improved by 28.583 >= min_delta = 0.0. New best score: 60.515
Metric val/total_cost improved by 11.307 >= min_delta = 0.0. New best score: 49.208
Metric val/total_cost improved by 7.248 >= min_delta = 0.0. New best score: 41.960
Metric val/total_cost improved by 1.744 >= min_delta = 0.0. New best score: 40.216
Metric val/total_cost improved by 0.862 >= min_delta = 0.0. New best score: 39.355
Metric val/total_cost improved by 0.439 >= min_delta = 0.0. New best score: 38.916
Metric val/total_cost improved by 0.774 >= min_delta = 0.0. New best score: 38.141
Metric val/total_cost improved by 0.347 >= min_delta = 0.0. New best score: 37.795
Metric val/total_cost improved by 0.200 >= min_delta = 0.0. New best score: 37.595
Metric val/total_cost improved by 0.196 >= min_delta = 0.0. New best score: 37.399
Metric val/total_cost improved by 0.194 >= min_delta = 0.0. New best score: 37.205
Monitored metric val/total_cos

### Let's test!

This loads the checkpoint with the lowest validation cost and runs evaluation on the test set.
Check example test results logged to Weights & Biases here: https://api.wandb.ai/links/cp_tobi/plk26iu9

In [80]:
# Example of how to correctly load a specific checkpoint for testing

# Replace this with the actual path to YOUR best checkpoint
#my_checkpoint_path = "mlpc2025-sed/mgw0942l/checkpoints/best-epoch=01.ckpt"

# You might need to re-initialize the model if you are in a new session
#model = SEDLightningModule.load_from_checkpoint(my_checkpoint_path)

# Then run test with the explicit path
#test_results = trainer.test(model, datamodule=dm, ckpt_path=my_checkpoint_path)


test_results = trainer.test(model, datamodule=dm, ckpt_path="best")   # test

Restoring states from the checkpoint path at .\mlpc2025-sed\6k652nc3\checkpoints\best-epoch=13.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at .\mlpc2025-sed\6k652nc3\checkpoints\best-epoch=13.ckpt
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃         Test metric         ┃        DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test/Chainsaw_bacc      │     0.9348557591438293      │
│      test/Chainsaw_f1       │     0.7624800205230713      │
│   test/Chainsaw_precision   │     0.6767581105232239      │
│    test/Chainsaw_recall     │     0.8730675578117371      │
│     test/Dog Bark_bacc      │     0.9065338969230652      │
│      test/Dog Bark_f1       │     0.7104825377464294      │
│   test/Dog Bark_precision   │     0.6248780488967896      │
│    test/Dog Bark_recall     │     0.8232647776603699      │
│     test/Horn Honk_bacc     │     0.7834319472312927      │
│      test/Horn Honk_f1      │     0.5248143672943115      │
│  test/Horn Honk_precision   │     0.48170730471611023     │
│    test/Horn Honk_recall    │     0.5763948559761047      │
│    test/Jackhammer_bacc     │     0.6887333393096924      │
│     test/Jackhammer_f1      │     0.3983474671840668      │
│  test/Jackhammer_precision  │     0.41655299067497253     │
│   test/Jackhammer_recall    │     0.3816666603088379      │
│    test/Lawn Mower_bacc     │     0.7060708403587341      │
│     test/Lawn Mower_f1      │     0.42924752831459045     │
│  test/Lawn Mower_precision  │     0.44352078437805176     │
│   test/Lawn Mower_recall    │     0.4158642888069153      │
│    test/Power Drill_bacc    │     0.8005107641220093      │
│     test/Power Drill_f1     │     0.3612326681613922      │
│ test/Power Drill_precision  │     0.25485655665397644     │
│   test/Power Drill_recall   │     0.6200312972068787      │
│   test/Rooster Crow_bacc    │     0.9004440903663635      │
│    test/Rooster Crow_f1     │     0.6806953549385071      │
│ test/Rooster Crow_precision │     0.5914149284362793      │
│  test/Rooster Crow_recall   │     0.8017241358757019      │
│       test/Shout_bacc       │     0.7962873578071594      │
│        test/Shout_f1        │     0.43473154306411743     │
│    test/Shout_precision     │     0.3363957703113556      │
│      test/Shout_recall      │     0.6143068075180054      │
│       test/Siren_bacc       │     0.9123786687850952      │
│        test/Siren_f1        │     0.7766689658164978      │
│    test/Siren_precision     │      0.727811336517334      │
│      test/Siren_recall      │     0.8325581550598145      │
│      test/Speech_bacc       │     0.9163296222686768      │
│       test/Speech_f1        │     0.7644153237342834      │
│    test/Speech_precision    │     0.6756536364555359      │
│     test/Speech_recall      │     0.8800257444381714      │
│     test/cost/Chainsaw      │     1.2901484966278076      │
│     test/cost/Dog Bark      │     1.2584105730056763      │
│     test/cost/Horn Honk     │      6.307921886444092      │
│    test/cost/Jackhammer     │     4.1989336013793945      │
│    test/cost/Lawn Mower     │     3.7514281272888184      │
│    test/cost/Power Drill    │      5.879459381103516      │
│   test/cost/Rooster Crow    │     0.1269518882036209      │
│       test/cost/Shout       │      5.957217216491699      │
│       test/cost/Siren       │      4.251301288604736      │
│      test/cost/Speech       │     5.1955060958862305      │
│          test/loss          │     1.0511618852615356      │
│       test/total_cost       │      38.21727752685547      │
└─────────────────────────────┴─────────────────────────────┘

## Compare Baseline, Logistic Regression and BiGRU Costs on Test Set

In [81]:
# baseline inference on test set
bl_total, bl_breakdown = evaluate_cost(
    test_files,
    DATASET_PATH,
    TARGET_CLASSES,
    X_test,
    bl_inference_funcs
)

In [82]:
# logistic regression inference on test set
lr_total, lr_breakdown = evaluate_cost(
    test_files,
    DATASET_PATH,
    TARGET_CLASSES,
    X_test,
    lr_inference_funcs
)

### Collect all costs in a pandas dataframe for pretty print

In [85]:
# shuffle around format for pretty print

# Convert breakdowns into dict[class → cost]
bl_costs = {cls: d["cost"] for cls, d in bl_breakdown.items()}
lr_costs = {cls: d["cost"] for cls, d in lr_breakdown.items()}

# Add total cost
bl_costs["TOTAL"] = bl_total
lr_costs["TOTAL"] = lr_total

# Extract relevant costs from pytorch lightning test results
rnn_result = test_results[0]
rnn_costs = {
    cls: rnn_result[f"test/cost/{cls}"]
    for cls in TARGET_CLASSES
    if f"test/cost/{cls}" in rnn_result
}

# Add total cost
rnn_costs["TOTAL"] = rnn_result["test/total_cost"]

# Create a DataFrame for comparison
cost_df = pd.DataFrame({
    "Baseline": bl_costs,
    "Logistic Regression": lr_costs,
    "RNN": rnn_costs
}).round(2)

In [86]:
print(tabulate(cost_df.reset_index().values,
               headers=["Class", "Baseline", "Logistic Regression", "RNN"],
               tablefmt="github"))

| Class        |   Baseline |   Logistic Regression |   RNN |
|--------------|------------|-----------------------|-------|
| Speech       |      27.05 |                  6.17 |  5.2  |
| Shout        |       9.92 |                  8.75 |  5.96 |
| Chainsaw     |       6.05 |                  2.07 |  1.29 |
| Jackhammer   |       6.36 |                  5.98 |  4.2  |
| Lawn Mower   |       5.36 |                  4.02 |  3.75 |
| Power Drill  |       8.24 |                  9.33 |  5.88 |
| Dog Bark     |       6.16 |                  2.04 |  1.26 |
| Rooster Crow |       0.48 |                  0.19 |  0.13 |
| Horn Honk    |      13.12 |                  9.18 |  6.31 |
| Siren        |      18.76 |                  5.97 |  4.25 |
| TOTAL        |     101.47 |                 53.69 | 38.22 |


## Compute predictions on customer's secret test set

Requires three functions:
- `load_model_from_checkpoint`: loading the desired model checkpoint, checkpoints are in folder `mlpc2025-sed/<wandb_id>/checkpoints`; an example checkpoint will be downloaded below
- `predict_dataset`: generate predictions for customer's dataset
- `segment_and_save`: bring predictions into the required 1.2 second segement format and save as csv file

In [92]:
def load_model_from_checkpoint(
    ckpt_path: str,
    hparams: dict,
    classes: list[str]
) -> pl.LightningModule:
    return SEDLightningModule.load_from_checkpoint(
        checkpoint_path=ckpt_path,
        input_dim  = hparams["input_dim"],
        hidden_dim = hparams["hidden_dim"],
        num_layers = hparams["num_layers"],
        lr         = hparams["lr"],
        threshold  = hparams["threshold"],
        classes    = classes,
        pos_weights = pos_weights
    )

In [93]:
def predict_dataset(
    model: pl.LightningModule,
    loader: DataLoader
) -> dict[str, dict[str, np.ndarray]]:
    """
    Runs trainer.predict() on `loader` and returns:
      preds_by_file[filename][class] = 1D NumPy array of frame‐wise {0,1}.
    """
    trainer = pl.Trainer(accelerator="auto", devices=1)
    outputs = trainer.predict(model, dataloaders=loader)

    # flatten into lists
    all_preds = {c: [] for c in model.classes}
    all_files = []
    for batch_out in outputs:
        for fname, pred in zip(batch_out["filenames"], batch_out["preds"]):
            all_files.append(fname)
            arr = pred.numpy()  # shape (T_i, C)
            for i, cls in enumerate(model.classes):
                all_preds[cls].append(arr[:, i])

    # repackage into preds_by_file
    preds_by_file: dict[str, dict[str, np.ndarray]] = {}
    for idx, fname in enumerate(all_files):
        preds_by_file.setdefault(fname, {})
        for cls in model.classes:
            preds_by_file[fname][cls] = all_preds[cls][idx]

    return preds_by_file

In [94]:
def segment_and_save(
    preds_by_file: dict[str, dict[str, np.ndarray]],
    class_names: list[str],
    dataset_path: str,
    out_csv: str,
    compute_cost: bool = False,
    test_files: list[str] = None,
) -> pd.DataFrame:
    """
    1) Build segment‐level DataFrame
    2) Sanity‐check with check_dataframe()
    3) (optional) compute & print cost if val_files is provided
    4) save CSV to out_csv
    """
    # 1) aggregate predictions using the function provided in compute_cost.py
    pred_df = get_segment_prediction_df(
        predictions = preds_by_file,
        class_names = class_names
    )

    # 2) sanity‐check (from compute_cost.py)
    check_dataframe(pred_df, dataset_path)

    # 3) cost (optional), for sanity check on our custom test split
    if compute_cost and test_files is not None:
        gt_df = get_ground_truth_df(test_files, dataset_path) # from compute_cost.py
        total, breakdown = total_cost(pred_df, gt_df) # from compute_cost.py
        print(f"\nTotal cost: {total:.4f}")

        gt_csv = os.path.splitext(out_csv)[0] + "_ground_truth.csv"
        gt_df.to_csv(gt_csv, index=False)
        print(f"Saved ground truth segments to {gt_csv}")

    # 4) save
    pred_df.to_csv(out_csv, index=False)
    print(f"Saved segment predictions to {out_csv}")

    return pred_df

### Load checkpoint

Download an example checkpoint from huggingface.

In [95]:
# download example checkpoint from huggingface
# ckpt_path = hf_hub_download(
#    repo_id="fschmid56/mlpc2025_dataset",
#    filename="colab_tutorial.ckpt",
#    repo_type="model"
#)

# alternatively, use your own local checkpoint
# replace wandb id 'lo9ygyg4' with your desired wandb id
# replace 'best-epoch=08.ckpt' with the name of your checkpoint
# ckpt_path = "/content/best-epoch=08.ckpt"
ckpt_path = "mlpc2025-sed/6k652nc3/checkpoints/best-epoch=13.ckpt"

model = load_model_from_checkpoint(ckpt_path, hparams, TARGET_CLASSES)

We sanity check model loading, the prediction routine and segmenting predictions by applying it our custom test split and calculating costs (as we have access to the labels).

In [96]:
# 1) TEST SPLIT
test_dataset = SequenceDataset(X_test, Y_test, TARGET_CLASSES, test_files)
test_loader  = DataLoader(test_dataset, batch_size=8, collate_fn=collate_fn)
test_preds   = predict_dataset(model, test_loader)
segment_and_save(
    preds_by_file = test_preds,
    class_names   = TARGET_CLASSES,
    dataset_path  = DATASET_PATH,
    out_csv       = "test_split_predictions.csv",
    compute_cost  = True,
    test_files     = test_files,
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]


Total cost: 38.2173
Saved ground truth segments to test_split_predictions_ground_truth.csv
Saved segment predictions to test_split_predictions.csv


,filename,onset,Speech,Shout,Chainsaw,Jackhammer,Lawn Mower,Power Drill,Dog Bark,Rooster Crow,Horn Honk,Siren
0,440698.mp3,0.0,0,0,0,0,0,0,0,0,0,0
1,440698.mp3,1.2,0,0,0,0,0,0,0,0,0,0
2,440698.mp3,2.4,0,0,0,0,0,0,0,0,0,0
3,440698.mp3,3.6,0,0,0,0,0,0,0,0,0,0
4,440698.mp3,4.8,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
31503,400159.mp3,19.2,0,0,0,0,0,0,0,0,0,0
31504,400159.mp3,20.4,0,0,0,0,0,0,0,0,0,0
31505,400159.mp3,21.6,0,0,0,0,0,0,0,0,0,0
31506,400159.mp3,22.8,0,0,0,0,0,0,0,0,0,0


Finally, compute predictions on the customer's secret test set and store as `/content/customer_predictions.csv`.

In [97]:
# 2) CUSTOMER SET (no labels → compute_cost=False)
customer_files = CUSTOMER_METADATA["filename"].unique()
X_cust, _ = read_files(customer_files, TARGET_CLASSES,
                       features_dir=CUSTOMER_AUDIO_FEATURES_DIR,
                       labels_dir=None)
cust_dataset = SequenceDataset(X_cust, None, TARGET_CLASSES, customer_files)
cust_loader  = DataLoader(cust_dataset, batch_size=8, collate_fn=collate_fn)

cust_preds = predict_dataset(model, cust_loader)
segment_and_save(
    preds_by_file = cust_preds,
    class_names   = TARGET_CLASSES,
    dataset_path  = CUSTOMER_DATASET_PATH,
    out_csv       = "customer_predictions.csv",
    compute_cost  = False,  # can't compute on customer's secret test set
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Saved segment predictions to customer_predictions.csv


,filename,onset,Speech,Shout,Chainsaw,Jackhammer,Lawn Mower,Power Drill,Dog Bark,Rooster Crow,Horn Honk,Siren
0,386984.mp3,0.0,1,1,0,0,0,0,0,0,0,0
1,386984.mp3,1.2,1,1,0,0,0,0,0,0,0,0
2,386984.mp3,2.4,1,1,0,0,0,0,0,0,0,0
3,386984.mp3,3.6,1,1,0,0,0,0,0,0,0,0
4,386984.mp3,4.8,1,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
52186,507531.mp3,21.6,0,0,0,0,0,0,0,0,0,0
52187,507531.mp3,22.8,0,0,0,0,0,0,0,0,0,0
52188,507531.mp3,24.0,0,0,0,0,0,0,0,0,0,0
52189,507531.mp3,25.2,0,0,0,0,0,0,0,0,0,0


### Final checks as in Task Description

Instead of importing all the functions from `compute_cost.py` and working with DataFrames directly, you can also run the provided script as recommended in the Task Description. Just pass your generated .csv file(s) to verify correctness and compute cost.

In [108]:
# !python compute_cost.py \
#   --dataset_path="{DATASET_PATH}" \
#   --ground_truth_csv="test_split_predictions_ground_truth.csv" \
#   --predictions_csv="test_split_predictions.csv"

# or for the customer's secret test set

Traceback (most recent call last):
  File "C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Projects\MLPC\MLPC Project Task 3\Challenge Tutorial\compute_cost.py", line 277, in <module>
    check_dataframe(df_pred, dataset_path=args.dataset_path)
  File "C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Projects\MLPC\MLPC Project Task 3\Challenge Tutorial\compute_cost.py", line 55, in check_dataframe

In [99]:
!python compute_cost.py \
  --dataset_path="{CUSTOMER_DATASET_PATH}" \
  --predictions_csv="customer_predictions.csv"

Predictions CSV formated correctly.


# More advanced approach

In [102]:
import numpy as np
from collections import defaultdict
import torch

# ==============================================================================
# PART 1: MODIFY THE MODEL'S PREDICT_STEP TO OUTPUT PROBABILITIES
# ==============================================================================
# The default predict_step applies a threshold. For tuning, we need the raw
# probabilities. We can dynamically replace the method for this run.

print("Step 1: Modifying the model to output probabilities...")

def predict_step_probabilities(self, batch, batch_idx):
    """A modified predict_step that returns probabilities instead of binary predictions."""
    if len(batch) == 4:
        X, _, lengths, filenames = batch
    else:
        X, lengths, filenames = batch

    # 1) Get raw logits from the model
    logits = self.model(X, lengths)

    # 2) Convert to probabilities using sigmoid
    probs  = torch.sigmoid(logits) # Output probabilities

    # 3) Remove padding and return
    batch_probs = [probs[b, :lengths[b]].cpu().numpy()
                   for b in range(X.size(0))]

    return {"filenames": filenames, "preds": batch_probs}

# Temporarily attach the new method to the class
SEDLightningModule.predict_step = predict_step_probabilities


# ==============================================================================
# PART 2: TUNE THE PIPELINE ON YOUR VALIDATION SET
# ==============================================================================
# We use the validation set to find the best thresholds and filter sizes.

print("\nStep 2: Tuning the pipeline on the validation set...")

# --- A: Generate probability predictions for the validation set ---
val_loader = dm.val_dataloader()
val_outputs = trainer.predict(model, dataloaders=val_loader)

# --- B: Reformat predictions into a {filename -> {class -> array}} dict ---
Y_pred_probs_val = defaultdict(dict)
for batch_out in val_outputs:
    for i, fname in enumerate(batch_out["filenames"]):
        # pred_array has shape (T_i, C)
        pred_array = batch_out["preds"][i]
        for j, cls in enumerate(TARGET_CLASSES):
            Y_pred_probs_val[fname][cls] = pred_array[:, j]

# --- C: Apply median filter to smooth probabilities ---
# (Tune kernel_size on your validation set; 7 is a reasonable start)
smoothed_probs_val = apply_median_filter(Y_pred_probs_val, TARGET_CLASSES, kernel_size=7)

# --- D: Tune thresholds to find the optimal values for each class ---
# Reformat data again for the tuning function: {class -> [array1, array2, ...]}
Y_pred_probs_reformatted = {c: [] for c in TARGET_CLASSES}
for fname in val_files: # val_files is from the train/val/test split
    if fname in smoothed_probs_val:
        for c in TARGET_CLASSES:
            Y_pred_probs_reformatted[c].append(smoothed_probs_val[fname][c])

# Y_val (ground truth) is also from the train/val/test split
optimal_thresholds = tune_thresholds(Y_val, Y_pred_probs_reformatted, TARGET_CLASSES)


# ==============================================================================
# PART 3: APPLY THE TUNED PIPELINE TO THE CUSTOMER TEST SET
# ==============================================================================
# Now we repeat the process on the secret test set, using the parameters
# we just found.

print("\nStep 3: Applying the tuned pipeline to the customer test set...")

# --- A: Load customer data and create a dataloader ---
customer_files = CUSTOMER_METADATA["filename"].unique()
X_cust, _ = read_files(customer_files, TARGET_CLASSES,
                       features_dir=CUSTOMER_AUDIO_FEATURES_DIR,
                       labels_dir=None)
cust_dataset = SequenceDataset(X_cust, None, TARGET_CLASSES, customer_files)
cust_loader  = DataLoader(cust_dataset, batch_size=hparams["batch_size"], collate_fn=collate_fn)

# --- B: Generate probability predictions for the customer set ---
cust_outputs = trainer.predict(model, dataloaders=cust_loader)

# --- C: Reformat predictions ---
Y_pred_probs_cust = defaultdict(dict)
for batch_out in cust_outputs:
    for i, fname in enumerate(batch_out["filenames"]):
        pred_array = batch_out["preds"][i]
        for j, cls in enumerate(TARGET_CLASSES):
            Y_pred_probs_cust[fname][cls] = pred_array[:, j]

# --- D: Apply the SAME median filter ---
smoothed_probs_cust = apply_median_filter(Y_pred_probs_cust, TARGET_CLASSES, kernel_size=7)

# --- E: Apply the OPTIMAL thresholds found during tuning ---
binary_preds_cust = defaultdict(dict)
for fname, class_probs in smoothed_probs_cust.items():
    for cls, probs in class_probs.items():
        threshold = optimal_thresholds[cls]
        binary_preds_cust[fname][cls] = (probs > threshold).astype(int)

# --- F: Apply duration filtering to the final binary predictions ---
# (Tune min_duration_frames on your validation set; 3 is a reasonable start)
cleaned_binary_preds_cust = apply_duration_filtering(binary_preds_cust, TARGET_CLASSES, min_duration_frames=3)


# ==============================================================================
# PART 4: SAVE THE FINAL SUBMISSION FILE
# ==============================================================================
print("\nStep 4: Saving the final predictions for submission...")

# The function `segment_and_save` aggregates the cleaned frame-level predictions
# into the required 1.2-second segments and saves the CSV file.
segment_and_save(
    preds_by_file = cleaned_binary_preds_cust,
    class_names   = TARGET_CLASSES,
    dataset_path  = CUSTOMER_DATASET_PATH,
    out_csv       = "customer_predictions.csv",
    compute_cost  = False, # Cannot compute cost on the secret test set
)

print("\nWorkflow complete. 'customer_predictions.csv' is ready for submission.")



Step 1: Modifying the model to output probabilities...

Step 2: Tuning the pipeline on the validation set...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Output()

Applying median filter with kernel size 7...
Tuning thresholds for each class...
 - Optimal threshold for Speech: 0.360 (cost: 26088.000)
 - Optimal threshold for Shout: 0.390 (cost: 27534.000)
 - Optimal threshold for Chainsaw: 0.330 (cost: 6168.000)
 - Optimal threshold for Jackhammer: 0.550 (cost: 25158.000)
 - Optimal threshold for Lawn Mower: 0.500 (cost: 22359.000)
 - Optimal threshold for Power Drill: 0.540 (cost: 47139.000)
 - Optimal threshold for Dog Bark: 0.290 (cost: 6156.000)
 - Optimal threshold for Rooster Crow: 0.490 (cost: 822.000)
 - Optimal threshold for Horn Honk: 0.590 (cost: 31215.000)
 - Optimal threshold for Siren: 0.430 (cost: 20751.000)

Step 3: Applying the tuned pipeline to the customer test set...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Applying median filter with kernel size 7...
Applying duration filtering with min duration 3 frames...

Step 4: Saving the final predictions for submission...
Saved segment predictions to customer_predictions.csv

Workflow complete. 'customer_predictions.csv' is ready for submission.


In [ ]:
!python compute_cost.py \
  --dataset_path="{CUSTOMER_DATASET_PATH}" \
  --predictions_csv="customer_predictions.csv"

# CALCULTING THE FINAL COST

In [109]:
print("--- Calculating Final Validation Cost ---")

# --- Step 1: Apply the optimal thresholds to the smoothed validation probabilities ---
# `smoothed_probs_val` should be available from the previous cell's execution.
final_binary_preds_val = defaultdict(dict)
for fname, class_probs in smoothed_probs_val.items():
    for cls, probs in class_probs.items():
        # Apply the specific threshold found for each class
        threshold = optimal_thresholds[cls]
        final_binary_preds_val[fname][cls] = (probs > threshold).astype(int)

# --- Step 2: Reformat predictions for the evaluate_cost function ---
# The function expects the format {class -> [array_file1, array_file2, ...]}
Y_pred_final_reformatted = {c: [] for c in TARGET_CLASSES}

# We must iterate through `val_files` to ensure the order is correct
for fname in val_files:
    if fname in final_binary_preds_val:
        for cls in TARGET_CLASSES:
            Y_pred_final_reformatted[cls].append(final_binary_preds_val[fname][cls])

# --- Step 3: Call the official evaluate_cost function and print the results ---
# This function is provided in your notebook
total, breakdown = evaluate_cost(
    val_files=val_files,
    dataset_path=DATASET_PATH,
    classes=TARGET_CLASSES,
    Y_pred=Y_pred_final_reformatted # Using our final, tuned predictions
)

print("\n" + "="*50)
print(f"✅ FINAL TOTAL COST ON VALIDATION SET: {total:.4f}")
print("="*50 + "\n")

# Optional: Display the cost breakdown per class
df = pd.DataFrame({cls: {"Avg. Cost per minute": round(m["cost"], 4)} for cls, m in breakdown.items()}).T
print("Cost Breakdown per Class:")
print(tabulate(df, headers="keys", tablefmt="github"))

--- Calculating Final Validation Cost ---

✅ FINAL TOTAL COST ON VALIDATION SET: 35.9523

Cost Breakdown per Class:
|              |   Avg. Cost per minute |
|--------------|------------------------|
| Speech       |                 4.3291 |
| Shout        |                 4.8808 |
| Chainsaw     |                 1.0541 |
| Jackhammer   |                 4.1781 |
| Lawn Mower   |                 3.7583 |
| Power Drill  |                 7.6884 |
| Dog Bark     |                 0.9825 |
| Rooster Crow |                 0.1622 |
| Horn Honk    |                 5.5135 |
| Siren        |                 3.4054 |


# Generate new `customer_predictions_opt.csv` file with optimal thresholds

In [116]:
print("--- Applying Final Pipeline to Customer Dataset for Submission ---")

# --- Step 1: Load customer data and create a dataloader ---
# This uses the predefined paths from the notebook
customer_files = CUSTOMER_METADATA["filename"].unique()
X_cust, _ = read_files(customer_files, TARGET_CLASSES,
                       features_dir=CUSTOMER_AUDIO_FEATURES_DIR,
                       labels_dir=None)
cust_dataset = SequenceDataset(X_cust, None, TARGET_CLASSES, customer_files)
cust_loader  = DataLoader(cust_dataset, batch_size=hparams["batch_size"], collate_fn=collate_fn, num_workers=0)


# --- Step 2: Generate probability predictions for the customer set ---
# Ensure your model is using the probability output version of predict_step
SEDLightningModule.predict_step = predict_step_probabilities
cust_outputs = trainer.predict(model, dataloaders=cust_loader)


# --- Step 3: Reformat predictions into a {filename -> {class -> array}} dict ---
Y_pred_probs_cust = defaultdict(dict)
for batch_out in cust_outputs:
    for i, fname in enumerate(batch_out["filenames"]):
        pred_array = batch_out["preds"][i]
        for j, cls in enumerate(TARGET_CLASSES):
            Y_pred_probs_cust[fname][cls] = pred_array[:, j]


# --- Step 4: Apply the same post-processing used during validation ---
# Using kernel_size = 7 as an example
smoothed_probs_cust = apply_median_filter(Y_pred_probs_cust, TARGET_CLASSES, kernel_size=7)


# --- Step 5: Apply the optimal thresholds found on the validation set ---
final_binary_preds_cust = defaultdict(dict)
for fname, class_probs in smoothed_probs_cust.items():
    for cls, probs in class_probs.items():
        # Using the optimal_thresholds dictionary we learned from the validation set
        threshold = optimal_thresholds[cls]
        final_binary_preds_cust[fname][cls] = (probs > threshold).astype(int)


# --- Step 6: (Optional) Apply final duration filtering ---
# Using min_duration_frames = 3 as an example
cleaned_binary_preds_cust = apply_duration_filtering(final_binary_preds_cust, TARGET_CLASSES, min_duration_frames=3)


# --- Step 7: Save the final predictions for submission ---
# This function aggregates predictions and saves them in the required CSV format
print("\nSaving final predictions to 'customer_predictions_opt.csv'...")
segment_and_save(
    preds_by_file = cleaned_binary_preds_cust,
    class_names   = TARGET_CLASSES,
    dataset_path  = CUSTOMER_DATASET_PATH,
    out_csv       = "customer_predictions_opt.csv",
    compute_cost  = False,  # MUST be False as we don't have customer labels
)

print("\n✅ 'customer_predictions_opt.csv' is ready for submission.")

--- Applying Final Pipeline to Customer Dataset for Submission ---


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Linus\Desktop\Studium\1. Semester\Python\3.11.7\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Output()

Applying median filter with kernel size 7...
Applying duration filtering with min duration 3 frames...

Saving final predictions to 'customer_predictions_opt.csv'...
Saved segment predictions to customer_predictions_opt.csv

✅ 'customer_predictions_opt.csv' is ready for submission.
